<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
%pip install beautifulsoup4 requests pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def clean_cell(cell):
    """Read a cell without citations, styles or scripts; keep all visible links."""
    fragment = BeautifulSoup(str(cell), 'html.parser')
    for annotation in fragment.select('sup, style, script'):
        annotation.decompose()
    text = unicodedata.normalize('NFKC', fragment.get_text(' ', strip=True))
    return re.sub(r'\s+', ' ', text).strip()

def date_time(table_cells):
    """Return the launch date and UTC time, excluding reference numbers."""
    text = clean_cell(table_cells)
    match = re.search(r'(\d{1,2} [A-Za-z]+ \d{4}),?\s*(\d{1,2}:\d{2}(?::\d{2})?)?', text)
    if not match:
        raise ValueError(f'Unexpected launch date: {text!r}')
    return [match.group(1), match.group(2) or pd.NA]

def booster_version(table_cells):
    return clean_cell(table_cells) or pd.NA

def landing_status(table_cells):
    return clean_cell(table_cells) or pd.NA

def get_mass(table_cells):
    """Keep the kg value when provided; do not turn unknown masses into zero."""
    text = clean_cell(table_cells)
    match = re.search(r'[~<>≤≥]?\s*\d[\d, .]*\s*kg', text)
    return match.group(0).strip() if match else (text or pd.NA)

def extract_column_from_header(row):
    """Extract a header without modifying the original HTML table."""
    name = clean_cell(row)
    name = re.sub(r'\(\s*UTC\s*\)', '(UTC)', name)
    if name == 'Version, Booster':
        name = 'Version Booster'
    return name if name and not name.isdigit() else None


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# Request the historical revision, checking the HTTP result before parsing it.
response = requests.get(static_url, headers=headers, timeout=60)
response.raise_for_status()


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(response.content, 'html.parser')
if soup.title is None or 'Falcon' not in soup.title.get_text():
    raise ValueError('The response is not the expected Falcon launch page.')


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
soup.title


<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all('table')


Starting from the third table is our target table contains the actual launch records.


In [9]:
# Locate the launch table by its header rather than relying on its position.
launch_tables = []
for table in html_tables:
    first_row = table.find('tr')
    if first_row is None:
        continue
    names = [extract_column_from_header(th) for th in first_row.find_all('th')]
    if 'Flight No.' in names and 'Payload mass' in names and 'Booster landing' in names:
        launch_tables.append(table)
if not launch_tables:
    raise ValueError('No launch tables found in the downloaded HTML.')
first_launch_table = launch_tables[0]
print(first_launch_table.prettify()[:4000])


<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
 <tbody id="mwmg">
  <tr id="mwmw">
   <th id="mwnA" scope="col">
    Flight No.
   </th>
   <th id="mwnQ" scope="col">
    Date and
    <br id="mwng"/>
    time (
    <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwoA" scope="col">
    <a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">
     Version,
     <br id="mwog"/>
     Booster
    </a>
    <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boosters are designated with 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
header_row = first_launch_table.find('tr')
for th in header_row.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)


Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time (UTC)', 'Version Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
# Replace the combined date/time header with two separate columns.
launch_dict = {name: [] for name in column_names if name != 'Date and time (UTC)'}
launch_dict['Date'] = []
launch_dict['Time'] = []


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
extracted_row = 0
# Each launch row has a flight-number header and nine data cells.
# Include FH 1, FH 2 and FH 3; skip the N/A prelaunch static-fire accident.
for table in launch_tables:
    for rows in table.find_all('tr'):
        heading = rows.find('th', recursive=False)
        if heading is None:
            continue
        flight_number = clean_cell(heading)
        if not re.fullmatch(r'(?:\d+|FH\s+\d+)', flight_number):
            continue
        row = rows.find_all('td', recursive=False)
        if len(row) != 9:
            raise ValueError(f'Flight {flight_number}: expected 9 cells, found {len(row)}')

        date, time = date_time(row[0])
        record = {
            'Flight No.': flight_number,
            'Date': date,
            'Time': time,
            'Version Booster': booster_version(row[1]),
            'Launch site': clean_cell(row[2]) or pd.NA,
            'Payload': clean_cell(row[3]) or pd.NA,
            'Payload mass': get_mass(row[4]),
            'Orbit': clean_cell(row[5]) or pd.NA,
            'Customer': clean_cell(row[6]) or pd.NA,
            'Launch outcome': clean_cell(row[7]) or pd.NA,
            'Booster landing': landing_status(row[8]),
        }
        for key in launch_dict:
            launch_dict[key].append(record[key])
        extracted_row += 1

print(f'Extracted {extracted_row} launch records from {len(launch_tables)} tables.')


Extracted 124 launch records from 9 tables.


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
# Equal-length lists ensure one complete record per dataframe row.
df = pd.DataFrame(launch_dict)
print(f'{df.shape[0]} rows, {df.shape[1]} columns')
df.head()


124 rows, 11 columns


,Flight No.,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Date,Time
0,1,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute),4 June 2010,18:45
1,2,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),NaN,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute),8 December 2010,15:43
2,3,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success,No attempt,22 May 2012,07:44
3,4,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt,8 October 2012,00:35
4,5,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt,1 March 2013,15:10


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [15]:
df.to_csv('spacex_web_scraped.csv', index=False)


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
